In [10]:
import pandas as pd
from src.data.raw_data import download_data
from src.data.process_data import deduplicate_categories
from src.utils.config import VALIDATED_DATA_PATH

In [11]:
data = download_data()
data.head(5)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher
0,3057270,Seafarer's Gambit,2024,"Jul 5, 2024",Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios
1,3822840,Capitalist Misadventures,2025,"Jul 25, 2025",Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios
2,3216640,The Beast and the Princess,2025,"Jun 17, 2025",Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames
3,2403620,Air Twister,2023,"Nov 10, 2023",Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ
4,1538040,Horde Slayer,2021,"Mar 19, 2021",Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues


In [12]:
# Validate nulls
null_before_clean = data.isnull().sum()
display('-Validation nulls count:-', null_before_clean)

# Handle nulls
data.dropna(subset=['genres', 'categories'], inplace=True)
data.fillna('Unknown', inplace=True)
data.reset_index(drop=True, inplace=True)

null_after_clean = data.isnull().sum()
display('-Handled nulls count:-', null_after_clean)

'-Validation nulls count:-'

appid                0
name                 0
release_year         0
release_date         0
genres              66
categories           7
price                0
recommendations      0
developer           53
publisher          183
dtype: int64

'-Handled nulls count:-'

appid              0
name               0
release_year       0
release_date       0
genres             0
categories         0
price              0
recommendations    0
developer          0
publisher          0
dtype: int64

In [13]:
# 'name' column Validation
data['name'] = data['name'].str.strip()

In [14]:
# 'release_year' and  'release_date' columns validation
print(f"Number of missing release years: {data['release_year'].isna().sum()}")
data_years = data['release_year'].unique()
study_years = [2021, 2022, 2023, 2024, 2025]
if sorted(data_years) != study_years:
    data = data[data['release_year'].isin(study_years)]
print(f"Unique years in 'release_year': {data_years}")

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
print(f"Number of missing release dates: {data['release_date'].isna().sum()}")
data.dropna(subset=['release_date'], inplace=True)
data.reset_index(drop=True, inplace=True)

Number of missing release years: 0
Unique years in 'release_year': [2024 2025 2023 2021 2022]
Number of missing release dates: 1399


In [15]:
# 'genres' and 'categories' columns validation
data = deduplicate_categories(data, 'genres')
data = deduplicate_categories(data, 'categories')
data.reset_index(drop=True, inplace=True)
data.shape

(63995, 10)

In [16]:
# 'price' and 'recommendations' columns validation
negative_prices = data['price'] < 0
negative_recommendations = data['recommendations'] < 0
data = data[~(negative_prices | negative_recommendations)]
print(f"Removed {negative_prices.sum()} negative prices and {negative_recommendations.sum()} negative recommendations")

Removed 0 negative prices and 0 negative recommendations


In [17]:
# 'developer' and 'publisher' columns validation
data['developer'] = data['developer'].str.strip()
data['publisher'] = data['publisher'].str.strip()

In [18]:
# Save validated data
print(f"Data final shape: {data.shape}")
data.to_parquet(f'{VALIDATED_DATA_PATH}', index=False)

Data final shape: (63995, 10)
